In [18]:
import os
import csv
import requests
import urllib
import zipfile
import threading
from tqdm.notebook import tqdm
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support import expected_conditions
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.common.by import By

BASE_URL = "https://qipedc.moet.gov.vn"
videos_dir = "Dataset/Videos"
text_dir = "Dataset/Text"

os.makedirs(videos_dir, exist_ok=True)
os.makedirs(text_dir, exist_ok=True)
csv_path = os.path.join(text_dir, "label.csv")
csv_lock = threading.Lock()

os.makedirs(videos_dir, exist_ok=True)
os.makedirs(text_dir, exist_ok=True)
csv_path = os.path.join(text_dir, "label.csv")
csv_lock = threading.Lock()

In [19]:
def download_chrome_driver():
    if not os.path.exists(chrome_driver_path):
        urllib.request.urlretrieve(chrome_driver_url, 'chromedriver-win64.zip')
        zip_ref = zipfile.ZipFile('chromedriver-win64.zip') 
        zip_ref.extractall()
        zip_ref.close()
        os.remove('chromedriver-win64.zip')

def handle_recursive_scrapping(data_list: list, driver):
    vids = WebDriverWait(driver=driver, timeout=3).until(
        expected_conditions.presence_of_all_elements_located((By.CSS_SELECTOR, "section:nth-of-type(2) > div:nth-of-type(2) > div:nth-of-type(1)"))
    )
    vids = driver.find_elements(By.CSS_SELECTOR, "section:nth-of-type(2) > div:nth-of-type(2) > div:nth-of-type(1) > a")
    for vid in vids:
        label = vid.find_element(By.CSS_SELECTOR, "p").text
        thumbs_url = vid.find_element(By.CSS_SELECTOR, "img").get_attribute("src")
        video_id = thumbs_url.replace("https://qipedc.moet.gov.vn/thumbs/", "").replace(".png", "")
        video_url = f"{BASE_URL}/videos/{video_id}.mp4"
        item = {'label': label, 'video_url': video_url}
        data_list.append(item)

def csv_init():
    if not os.path.exists(csv_path):
        with open(csv_path, 'w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(["ID", "VIDEO", "LABEL"])

def add_to_csv(id_val, video, label):
    with csv_lock:
        with open(csv_path, 'a', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow([id_val, video, label])

def download_video(video_data):
    video_url = video_data.get('video_url')
    label = video_data.get('label')
    filename = os.path.basename(urlparse(video_url).path)
    output_path = os.path.join(videos_dir, filename)
    if os.path.exists(output_path):
        print(f"Skip: {filename}")
        return
    try:
        print(f"Downloading: {filename}")
        response = requests.get(video_url, stream=True)
        response.raise_for_status()
        total_size = int(response.headers.get('content-length', 0))
        with open(output_path, 'wb') as file, tqdm(
            desc=f"Progess {filename}",
            total=total_size,
            unit='B',
            unit_scale=True,
            unit_divisor=1024,
            ncols=100
        ) as bar:
            for data in response.iter_content(chunk_size=8192):
                size = file.write(data)
                bar.update(size)
        id_val = sum(1 for _ in open(csv_path, encoding='utf-8'))
        add_to_csv(id_val, filename, label)                  
        print(f"Completed: {filename}")
        print(f"Updated label.csv: {label}")
    except Exception as e:
        print(f"Error {filename}: {str(e)}")
        if os.path.exists(output_path):
            os.remove(output_path)  

In [20]:
def crawl_videos():
    print("CRAWLING VIDEOS")
    options = Options()
    options.add_argument('--disable-dev-shm-usage')
    
    # Initialize without a specific Service path
    # Selenium Manager will auto-download the correct driver for Chrome v149
    driver = webdriver.Chrome(options=options)
    
    videos = []
    try:
        driver.get("https://qipedc.moet.gov.vn/dictionary")
        print("Connected to dictionary website")
        handle_recursive_scrapping(videos, driver)
        for i in range(2, 5):
            btn_id = i if i == 2 else i + 1
            button = driver.find_element(By.CSS_SELECTOR, f"button:nth-of-type({btn_id})")
            button.click()
            handle_recursive_scrapping(videos, driver)
        for i in range(5, 218):
            button = driver.find_element(By.CSS_SELECTOR, "button:nth-of-type(6)")
            button.click()
            handle_recursive_scrapping(videos, driver)
        for i in range(218, 220):
            btn_id = 6 if i == 218 else 7
            button = driver.find_element(By.CSS_SELECTOR, f"button:nth-of-type({btn_id})")
            button.click()
            handle_recursive_scrapping(videos, driver)
    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.close()
        return videos

In [21]:
download_chrome_driver()
videos = crawl_videos()

if videos:
    print(f"Found {len(videos)} videos\n")

print("STARTING DOWNLOAD VIDEOS")
csv_init()

if not videos:
    print("Videos not found")
else:
    with ThreadPoolExecutor(max_workers=3) as executor:
        executor.map(download_video, videos)
        
    print(f"DOWNLOAD COMPLETED {videos_dir}")

CRAWLING VIDEOS
Connected to dictionary website
Error: 'NoneType' object has no attribute 'text'


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=149.0.7827.115)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff68e4a3fa5+14925]
	chromedriver!GetHandleVerifier [0x7ff68e4a4000+14980]
	chromedriver!(No symbol) [0x7ff68dfe793d]
	chromedriver!(No symbol) [0x7ff68dfbe362]
	chromedriver!(No symbol) [0x7ff68e071846]
	chromedriver!(No symbol) [0x7ff68e07ba01]
	chromedriver!(No symbol) [0x7ff68e03401c]
	chromedriver!(No symbol) [0x7ff68e034f43]
	chromedriver!GetHandleVerifier [0x7ff68ea87591+5f7f11]
	chromedriver!GetHandleVerifier [0x7ff68ea81902+5f2282]
	chromedriver!GetHandleVerifier [0x7ff68eaa7115+617a95]
	chromedriver!GetHandleVerifier [0x7ff68e4c1dce+3274e]
	chromedriver!GetHandleVerifier [0x7ff68e4ca82c+3b1ac]
	chromedriver!GetHandleVerifier [0x7ff68e4ad744+1e0c4]
	chromedriver!GetHandleVerifier [0x7ff68e4ad8d4+1e254]
	chromedriver!GetHandleVerifier [0x7ff68e491447+1dc7]
	KERNEL32!BaseThreadIn